# 02 — Data Preprocessing

This notebook prepares the raw review dataset for machine learning.

The main objectives are to inspect duplicate reviews, validate the target labels,
perform necessary cleaning, and create training and testing datasets while
preventing data leakage.

In [8]:
import numpy as np
import pandas as pd

from pathlib import Path

# Define the dataset path
DATA_PATH = Path("../data/raw/fake reviews dataset.csv")

# Load dataset
df = pd.read_csv(DATA_PATH)

# Display first 5 rows
df.head()

df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)

Original dataset shape: (40432, 4)


In [13]:
df_processed = df.copy()

# Remove leading and trailing whitespace
df_processed["text_"] = (
    df_processed["text_"]
    .astype("string")
    .str.strip()
)

# Count duplicate review texts after normalization
duplicate_count = df_processed["text_"].duplicated().sum()

print("Duplicate review texts to remove:", duplicate_count)

# Remove duplicate review texts
df_processed = df_processed.drop_duplicates(
    subset=["text_"],
    keep="first"
).copy()

print("Dataset shape after duplicate removal:", df_processed.shape)

Duplicate review texts to remove: 22
Dataset shape after duplicate removal: (40410, 4)


In [10]:
label_mapping = {
    "OR": 0,
    "CG": 1
}

df_processed["target"] = df_processed["label"].map(label_mapping)

# Verify that every label was mapped
assert df_processed["target"].notna().all()

df_processed["target"] = df_processed["target"].astype(int)

print(df_processed[["label", "target"]].drop_duplicates())
print("\nTarget distribution:")
print(df_processed["target"].value_counts())

   label  target
0     CG       1
55    OR       0

Target distribution:
target
0    20215
1    20195
Name: count, dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

# Features and target
X = df_processed["text_"]
y = df_processed["target"]

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting label distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training samples: 32328
Testing samples: 8082

Training label distribution:
target
0    50.024746
1    49.975254
Name: proportion, dtype: float64

Testing label distribution:
target
0    50.024746
1    49.975254
Name: proportion, dtype: float64


In [12]:
# Create processed data directory


# Save train and test datasets
train_df = pd.DataFrame({
    "text": X_train,
    "target": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "target": y_test
})

train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print("Training and testing datasets saved successfully.")

Training and testing datasets saved successfully.


In [14]:
# Check for missing values
print("Missing values in training data:")
print(train_df.isnull().sum())

print("\nMissing values in testing data:")
print(test_df.isnull().sum())

# Check for duplicate reviews within each dataset
print("\nDuplicate training reviews:",
      train_df["text"].duplicated().sum())

print("Duplicate testing reviews:",
      test_df["text"].duplicated().sum())

# Check for overlap between training and testing reviews
train_texts = set(train_df["text"])
test_texts = set(test_df["text"])

overlap = train_texts.intersection(test_texts)

print("\nReviews appearing in both sets:", len(overlap))

Missing values in training data:
text      0
target    0
dtype: int64

Missing values in testing data:
text      0
target    0
dtype: int64

Duplicate training reviews: 0
Duplicate testing reviews: 0

Reviews appearing in both sets: 0


### Handling Duplicate Reviews

The original dataset contained 20 duplicate review-text occurrences. After removing leading and trailing whitespace, this increased to 22 duplicate occurrences.

We removed repeated review texts while retaining one occurrence of each unique review. This decision was made for two reasons:

1. To prevent identical reviews from appearing in both training and testing sets, which could lead to data leakage and overly optimistic evaluation results.
2. To prevent repeated records from giving certain review texts disproportionate influence during model training.

Some duplicated review texts had different ratings, but none had conflicting CG/OR labels. Since our primary classification task uses review text rather than ratings, we retained the first occurrence of each unique review text.